# Delivery Route Optimization

This notebook explores delivery locations and grocery store locations in Washington, DC, as a first step toward finding optimal delivery routes.

**Data**
- `data/route_optimization/Delivery_Locations.geojson` — delivery destination points
- `data/route_optimization/Grocery_Store_Locations.geojson` — grocery store points (potential route origins/depots)

In [1]:
import geopandas as gpd
import folium
from pathlib import Path

## Load Data

In [2]:
data_dir = Path('../data/route_optimization')

deliveries = gpd.read_file(data_dir / 'Delivery_Locations.geojson')
stores = gpd.read_file(data_dir / 'Grocery_Store_Locations.geojson')

print(f'Delivery locations: {len(deliveries)} points, CRS: {deliveries.crs}')
print(f'Grocery stores: {len(stores)} points, CRS: {stores.crs}')

Delivery locations: 100 points, CRS: EPSG:4326
Grocery stores: 14 points, CRS: EPSG:4326


In [3]:
deliveries.head()

,X,Y,OBJECTID,MAR_ID,ADDRESS,ADDRESS_NUMBER,ADDRESS_NUMBER_SUFFIX,STREET_NAME,STREET_TYPE,QUADRANT,...,BEGIN_DATE,BEGIN_DATE_SOURCE,FIRST_KNOWN_DATE,FIRST_KNOWN_DATE_SOURCE,CREATED_DATE,LAST_EDITED_DATE,SE_ANNO_CAD_DATA,SMD,ANC,geometry
0,-8.572740e+06,4.709399e+06,190522676,227074,31 S STREET NW,31,NaN,S,STREET,NW,...,1898-07-05 00:00:01+00:00,BUILDING PERMIT,None,None,2005-05-12 00:00:00+00:00,2008-01-17 10:05:45+00:00,None,SMD 5E03,ANC 5E,POINT (-77.01023 38.91427)
1,-8.566997e+06,4.701735e+06,190523950,48460,2330 BRANCH AVENUE SE,2330,NaN,BRANCH,AVENUE,SE,...,1928-07-24 00:00:01+00:00,BUILDING PERMIT,None,None,2005-05-12 00:00:00+00:00,NaT,None,SMD 7B06,ANC 7B,POINT (-76.95864 38.86067)
2,-8.579933e+06,4.710210e+06,190524019,270167,2325 HUIDEKOPER PLACE NW,2325,NaN,HUIDEKOPER,PLACE,NW,...,1938-07-11 00:00:01+00:00,BUILDING PERMIT,None,None,2005-05-12 00:00:00+00:00,NaT,None,SMD 3B05,ANC 3B,POINT (-77.07485 38.91993)
3,-8.577815e+06,4.707264e+06,190524033,279606,2650 VIRGINIA AVENUE NW,2650,NaN,VIRGINIA,AVENUE,NW,...,1967-01-01 00:00:01+00:00,OP HISTORIC BUILDING DATA,None,None,2011-05-27 10:31:40+00:00,2011-05-27 10:31:40+00:00,None,SMD 2A04,ANC 2A,POINT (-77.05582 38.89934)
4,-8.569213e+06,4.707131e+06,190526068,286448,1729 GALES PLACE NE,1729,NaN,GALES,PLACE,NE,...,1949-03-17 00:00:01+00:00,BUILDING PERMIT,None,None,2005-05-12 00:00:00+00:00,NaT,None,SMD 7D06,ANC 7D,POINT (-76.97854 38.89841)


In [4]:
stores.head()

,STORENAME,ADDRESS,ZIPCODE,PHONE,WARD,SSL,NOTES,PRESENT90,PRESENT95,PRESENT00,...,PRESENT25,XCOORD,YCOORD,MAR_ID,SE_ANNO_CAD_DATA,GLOBALID,CREATED,EDITED,OBJECTID,geometry
0,Safeway,5545 CONNECTICUT AVENUE NW,20015,2022446097,Ward 3,1867 0092,,No,Yes,Yes,...,Yes,393508.13,144078.34,263999,None,{902F540E-FE16-43A5-8010-CD84E94B28B7},2022-09-28 22:04:54+00:00,2025-01-14 19:33:19+00:00,645,POINT (-77.07491 38.9646)
1,Safeway,1747 COLUMBIA ROAD NW,20009,2026670774,Ward 1,2580 0512,,Yes,Yes,Yes,...,Yes,396445.44,139593.32,284083,None,{7C1459D3-4205-465C-8589-CDC4A462B587},2022-09-28 22:04:54+00:00,2025-01-14 19:33:19+00:00,667,POINT (-77.04099 38.92422)
2,Safeway,4865 MACARTHUR BOULEVARD NW,20007,2023375649,Ward 3,1389 0025,,No,Yes,Yes,...,No,391678.11,138844.95,285004,None,{B781FF71-032E-4647-86D6-214D329796D9},2022-09-28 22:04:54+00:00,2025-01-14 19:33:19+00:00,676,POINT (-77.09596 38.91744)
3,Safeway,1855 WISCONSIN AVENUE NW,20007,2023333223,Ward 2,1299 1040,,Yes,No,No,...,Yes,394141.03,138696.71,224665,None,{152A7BF1-F352-41DA-8C39-350BB15C9DA0},2022-09-28 22:04:54+00:00,2025-01-14 19:33:19+00:00,679,POINT (-77.06756 38.91613)
4,Safeway,1701 CORCORAN STREET NW,20009,2026676825,Ward 2,0155 0231,,Yes,Yes,Yes,...,Yes,396621.59,138243.64,241699,None,{490DC6A4-034E-4AF9-ACD1-569A11F60998},2022-09-28 22:04:54+00:00,2025-01-14 19:33:19+00:00,682,POINT (-77.03896 38.91206)


## Interactive Map

In [5]:
m = stores.explore(
    color='red',
    marker_kwds={'radius': 6},
    tooltip='STORENAME',
    name='Grocery Stores',
)

deliveries.explore(
    m=m,
    color='blue',
    marker_kwds={'radius': 3},
    tooltip='ADDRESS',
    name='Delivery Locations',
)

folium.LayerControl().add_to(m)
m

## Route Optimization

We need to serve all 100 deliveries from the 14 grocery stores, using round trips of exactly 10 stops (store -> 10 deliveries -> same store), with each store limited to 5 trips/day, minimizing total distance traveled. This is a multi-depot capacitated vehicle routing problem, solved using the OpenRouteService (ORS) Optimization API (built on the VROOM solver):

1. **Get distances** from every store to every delivery with one ORS Matrix call.
2. **Decide how many trips each store runs** with a local search that directly minimizes total assignment distance (not a proxy), **assign every delivery to a store** optimally given that plan, then **group each store's deliveries into compact batches of 10**. Two earlier, simpler versions of this step were tried and rejected: pure geography-first batching left several nearby stores idle while others served needlessly long trips, and even a greedy "store proposes its cheapest batch each round" version still left the same stores idle and produced one 73 km outlier trip, because a diagnostic showed one idle store was the genuine nearest store for several deliveries (by up to 6.3 km) yet never won a greedy round. The local-search version fixed this: total assignment distance dropped from ~264-271 km to 237 km and the worst-case gap dropped to 2.6 km.
3. **Route** each trip with ORS Optimization (1 vehicle + 10 jobs per call) to get the optimal stop order and real road-route geometry.

ORS's public API enforces per-request limits (max 3 vehicles / 70 locations for Optimization, max 3,500 distance pairs for Matrix) that make a single request covering all 14 stores and 100 deliveries infeasible — this design keeps every individual call comfortably within those limits.

In [6]:
import time

import numpy as np
import pandas as pd
import requests
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

### API Key

Note: the key below is stored in plaintext in this notebook file. Rotate/protect it if this notebook is ever shared or this folder is put under version control.

In [ ]:
ORS_API_KEY = '<ADD_YOUR_ORS_API_KEY>'
ORS_HEADERS = {'Authorization': ORS_API_KEY, 'Content-Type': 'application/json'}

ORS_MATRIX_URL = 'https://api.openrouteservice.org/v2/matrix/driving-car'
ORS_OPTIMIZATION_URL = 'https://api.openrouteservice.org/optimization'

In [8]:
def decode_polyline(encoded, precision=5):
    """Decode an ORS-encoded polyline into a list of [lon, lat] pairs.

    The ORS Optimization endpoint only includes `distance`/`geometry` in its
    response when the request body sets `"options": {"g": true}` -- the
    top-level `"geometry": true` field shown in some examples returns null
    for both on this account (confirmed by testing).
    """
    inv = 10 ** precision
    coords = []
    index = lat = lng = 0
    while index < len(encoded):
        for is_lat in (True, False):
            shift = result = 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1f) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = ~(result >> 1) if (result & 1) else (result >> 1)
            if is_lat:
                lat += delta
            else:
                lng += delta
        coords.append([lng / inv, lat / inv])
    return coords

### Step 1 — Get store-to-delivery road distances

Get real road distances from every store to every delivery with a single ORS Matrix call (14 x 100 = 1,400 pairs, under the 3,500-pair limit).

In [9]:
n_stores = len(stores)
n_deliveries = len(deliveries)

store_coords = [[x, y] for x, y in zip(stores.geometry.x, stores.geometry.y)]
delivery_coords = [[x, y] for x, y in zip(deliveries.geometry.x, deliveries.geometry.y)]

matrix_body = {
    'locations': store_coords + delivery_coords,
    'sources': list(range(n_stores)),
    'destinations': list(range(n_stores, n_stores + n_deliveries)),
    'metrics': ['distance'],
}
response = requests.post(ORS_MATRIX_URL, json=matrix_body, headers=ORS_HEADERS, timeout=60)
response.raise_for_status()

store_delivery_distances = np.array(response.json()['distances'])  # meters, shape (n_stores, n_deliveries)
print(f'Matrix shape: {store_delivery_distances.shape}')

Matrix shape: (14, 100)


### Step 2 — Decide how many trips each store runs (local search)

Start from an even trip allocation across stores, then repeatedly try moving one trip-slot from one store to another (respecting the 5-trip cap) and keep the move only if it lowers the total store-to-delivery assignment distance — solved exactly via the Hungarian algorithm for each candidate. Repeat until no single move helps. This optimizes the real objective directly, so a store only stays idle when giving it a trip genuinely wouldn't reduce total distance, not because of the order batches happened to form in.

In [10]:
TRIPS_PER_STORE = 5
BATCH_SIZE = 10
N_TRIPS_TOTAL = n_deliveries // BATCH_SIZE


def solve_assignment(n_trips):
    # Each store contributes n_trips[store] * 10 identical-cost columns, so the
    # matrix always has exactly n_deliveries columns -- a perfect (optimal) matching.
    store_columns = np.repeat(np.arange(n_stores), n_trips * BATCH_SIZE)
    cost_matrix = store_delivery_distances[store_columns].T
    delivery_idx, slot_idx = linear_sum_assignment(cost_matrix)
    total_cost = cost_matrix[delivery_idx, slot_idx].sum()
    delivery_to_store = {int(d): int(store_columns[s]) for d, s in zip(delivery_idx, slot_idx)}
    return total_cost, delivery_to_store


n_trips = np.zeros(n_stores, dtype=int)
for i in range(N_TRIPS_TOTAL):
    n_trips[i % n_stores] += 1

current_cost, _ = solve_assignment(n_trips)

improved = True
while improved:
    improved = False
    for donor in range(n_stores):
        if n_trips[donor] == 0:
            continue
        for receiver in range(n_stores):
            if receiver == donor or n_trips[receiver] >= TRIPS_PER_STORE:
                continue
            trial = n_trips.copy()
            trial[donor] -= 1
            trial[receiver] += 1
            trial_cost, _ = solve_assignment(trial)
            if trial_cost < current_cost - 1e-6:
                n_trips, current_cost, improved = trial, trial_cost, True
                break
        if improved:
            break

print(f'Trip allocation converged. Total assignment distance: {current_cost / 1000:.2f} km')

Trip allocation converged. Total assignment distance: 237.18 km


### Step 2b — Assign every delivery to a store

Given the final trip allocation from the search, run the exact assignment once more to get each delivery's serving store.

In [11]:
final_cost, delivery_to_store = solve_assignment(n_trips)
print(f'Final total assignment distance: {final_cost / 1000:.2f} km')

Final total assignment distance: 237.18 km


### Step 2c — Split each store's deliveries into compact trips of 10

Store choice is already decided optimally. For stores running more than one trip, this only groups their assigned deliveries into geographically compact batches of 10 (nearest-neighbor chaining on a projected CRS), so each trip's stops are close together for efficient in-trip routing.

In [12]:
deliveries_proj = deliveries.to_crs('EPSG:32618')
proj_coords = np.column_stack([deliveries_proj.geometry.x, deliveries_proj.geometry.y])

trips = []
for store_idx in range(n_stores):
    store_deliveries = [d for d, s in delivery_to_store.items() if s == store_idx]
    remaining_local = list(store_deliveries)
    while remaining_local:
        seed = remaining_local[0]
        dists = cdist([proj_coords[seed]], proj_coords[remaining_local])[0]
        nearest = [remaining_local[i] for i in np.argsort(dists)[:BATCH_SIZE]]
        trips.append({'store_idx': store_idx, 'deliveries': nearest})
        remaining_local = [d for d in remaining_local if d not in nearest]

print(f'Formed {len(trips)} trips')

Formed 10 trips


In [13]:
trips_per_store = pd.Series([t['store_idx'] for t in trips]).value_counts()

assert len(trips) == n_deliveries // BATCH_SIZE
assert sorted(d for t in trips for d in t['deliveries']) == list(range(n_deliveries))
assert trips_per_store.max() <= TRIPS_PER_STORE

trip_summary_df = pd.DataFrame({
    'trip_idx': range(len(trips)),
    'store': [f"{stores.iloc[t['store_idx']]['STORENAME']} ({stores.iloc[t['store_idx']]['ADDRESS']})" for t in trips],
    'store_idx': [t['store_idx'] for t in trips],
    'batch_cost_km': [round(store_delivery_distances[t['store_idx'], t['deliveries']].sum() / 1000, 2) for t in trips],
})
trip_summary_df

,trip_idx,store,store_idx,batch_cost_km
0,0,Safeway (1855 WISCONSIN AVENUE NW),3,21.61
1,1,Safeway (1701 CORCORAN STREET NW),4,20.89
2,2,Safeway (1601 MARYLAND AVENUE NE),5,7.98
3,3,Safeway (1601 MARYLAND AVENUE NE),5,22.74
4,4,Safeway (415 14TH STREET SE),6,34.07
5,5,Safeway (2845 ALABAMA AVENUE SE),8,18.56
6,6,Safeway (6500 PINEY BRANCH ROAD NW),9,31.37
7,7,Safeway (4203 DAVENPORT STREET NW),10,29.57
8,8,Safeway (3830 GEORGIA AVENUE NW),11,22.05
9,9,Safeway (322 40TH STREET NE),13,28.35


### Step 3 — Solve optimal stop order per trip (ORS Optimization)

For each of the 10 trips, call ORS Optimization with exactly 1 vehicle and 10 jobs (11 locations) — comfortably within the 3-vehicle / 70-location limits — to get the exact optimal visiting order and real road-route geometry for that trip.

In [14]:
trip_results = []
for trip_idx, trip in enumerate(trips):
    store_idx = trip['store_idx']
    store_lon, store_lat = store_coords[store_idx]

    jobs = [
        {'id': int(d_idx), 'location': delivery_coords[d_idx], 'amount': [1]}
        for d_idx in trip['deliveries']
    ]
    vehicles = [{
        'id': 0,
        'profile': 'driving-car',
        'start': [store_lon, store_lat],
        'end': [store_lon, store_lat],
        'capacity': [10],
    }]

    body = {'jobs': jobs, 'vehicles': vehicles, 'options': {'g': True}}
    response = requests.post(ORS_OPTIMIZATION_URL, json=body, headers=ORS_HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()

    if data['unassigned']:
        print(f'Trip {trip_idx}: {len(data["unassigned"])} unassigned jobs')

    trip_results.append({'trip_idx': trip_idx, 'store_idx': store_idx, 'route': data['routes'][0]})
    time.sleep(1)

print(f'Solved {len(trip_results)} trips')

Solved 10 trips


### Step 4 — Build the trip schedule table

In [15]:
address_lookup = deliveries['ADDRESS'].to_dict()

trip_counter = {}
schedule_rows = []
for result in trip_results:
    store_idx = result['store_idx']
    trip_counter[store_idx] = trip_counter.get(store_idx, 0) + 1
    result['trip'] = trip_counter[store_idx]

    route = result['route']
    stops = [address_lookup[step['id']] for step in route['steps'] if step['type'] == 'job']

    # STORENAME alone isn't unique (multiple stores share the same chain name),
    # so store_idx identifies the specific store and its address disambiguates it for display.
    schedule_rows.append({
        'store_idx': store_idx,
        'store': f"{stores.iloc[store_idx]['STORENAME']} ({stores.iloc[store_idx]['ADDRESS']})",
        'trip': result['trip'],
        'stops': stops,
        'n_stops': len(stops),
        'distance_km': round(route['distance'] / 1000, 2),
        'duration_min': round(route['duration'] / 60, 1),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(['store_idx', 'trip']).reset_index(drop=True)

assert schedule_df['n_stops'].sum() == 100
assert schedule_df.groupby('store_idx')['trip'].count().max() <= TRIPS_PER_STORE

schedule_df.drop(columns='store_idx')

,store,trip,stops,n_stops,distance_km,duration_min
0,Safeway (1855 WISCONSIN AVENUE NW),1,"[1556 34TH STREET NW, 1528 29TH STREET NW, 265...",10,19.54,50.0
1,Safeway (1701 CORCORAN STREET NW),1,"[1749 18TH STREET NW, 2292 CHAMPLAIN STREET NW...",10,17.14,37.9
2,Safeway (1601 MARYLAND AVENUE NE),1,"[830 18TH STREET NE, 844 19TH STREET NE, 2000 ...",10,7.19,18.6
3,Safeway (1601 MARYLAND AVENUE NE),2,"[1741 A STREET SE, 1440 INDEPENDENCE AVENUE SE...",10,17.97,39.8
4,Safeway (415 14TH STREET SE),1,"[751 12TH STREET SE, 1306 S STREET SE, 1517 V ...",10,21.70,48.8
5,Safeway (2845 ALABAMA AVENUE SE),1,"[2330 BRANCH AVENUE SE, 2601 33RD STREET SE, 3...",10,12.45,31.5
6,Safeway (6500 PINEY BRANCH ROAD NW),1,"[4808 16TH STREET NW, 5331 COLORADO AVENUE NW,...",10,14.88,35.7
7,Safeway (4203 DAVENPORT STREET NW),1,"[4460 SEDGWICK STREET NW, 4509 FOXHALL CRESCEN...",10,21.75,51.5
8,Safeway (3830 GEORGIA AVENUE NW),1,"[4213 7TH STREET NW, 4834 ILLINOIS AVENUE NW, ...",10,15.10,35.5
9,Safeway (322 40TH STREET NE),1,"[4605 CENTRAL AVENUE NE, 423 CHAPLIN STREET SE...",10,20.11,45.6


## Optimized Routes Map

In [16]:
m = stores.explore(
    color='red',
    marker_kwds={'radius': 6},
    tooltip='STORENAME',
    name='Grocery Stores',
)

deliveries.explore(
    m=m,
    color='blue',
    marker_kwds={'radius': 3},
    tooltip='ADDRESS',
    name='Delivery Locations',
)

trip_colors = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00',
    '#a65628', '#f781bf', '#999999', '#66c2a5', '#1b9e77',
]

routes_group = folium.FeatureGroup(name='Optimized Routes')
for i, result in enumerate(trip_results):
    route = result['route']
    store_label = f"{stores.iloc[result['store_idx']]['STORENAME']} ({stores.iloc[result['store_idx']]['ADDRESS']})"

    # decode_polyline returns [lon, lat]; Folium needs [lat, lon]
    lonlat_coords = decode_polyline(route['geometry'])
    latlon_coords = [[lat, lon] for lon, lat in lonlat_coords]

    folium.PolyLine(
        locations=latlon_coords,
        color=trip_colors[i % len(trip_colors)],
        weight=4,
        tooltip=f"{store_label} Trip {result['trip']}: {route['distance'] / 1000:.1f} km",
    ).add_to(routes_group)

routes_group.add_to(m)
folium.LayerControl().add_to(m)
m

## Export Routes to GeoJSON

Save all trips as a single GeoJSON file for validation in a GIS tool: one LineString feature per route, plus a Point feature for each stop (store start, each delivery in visit order, store end), tagged with `trip` and `store` so they can be filtered or styled per trip.

In [17]:
from shapely.geometry import LineString, Point

output_dir = Path('../outputs/route_optimization')
output_dir.mkdir(parents=True, exist_ok=True)

rows = []
for result in trip_results:
    store_idx = result['store_idx']
    store_row = stores.iloc[store_idx]
    store_label = f"{store_row['STORENAME']} ({store_row['ADDRESS']})"
    route = result['route']

    rows.append({
        'feature_type': 'route',
        'store': store_label,
        'trip': result['trip'],
        'sequence': None,
        'address': None,
        'distance_km': round(route['distance'] / 1000, 2),
        'duration_min': round(route['duration'] / 60, 1),
        'geometry': LineString(decode_polyline(route['geometry'])),
    })

    for sequence, step in enumerate(route['steps']):
        if step['type'] == 'job':
            feature_type, address = 'stop_delivery', address_lookup[step['id']]
        else:
            feature_type, address = 'stop_store', store_row['ADDRESS']
        rows.append({
            'feature_type': feature_type,
            'store': store_label,
            'trip': result['trip'],
            'sequence': sequence,
            'address': address,
            'distance_km': None,
            'duration_min': None,
            'geometry': Point(step['location']),
        })

routes_gdf = gpd.GeoDataFrame(rows, crs='EPSG:4326')
output_path = output_dir / 'routes.geojson'
routes_gdf.to_file(output_path, driver='GeoJSON')

print(f'Saved {len(trip_results)} trips ({len(routes_gdf)} features) to {output_path.resolve()}')

Saved 10 trips (130 features) to C:\Users\ujava\Desktop\claude_code_geospatial\outputs\route_optimization\routes.geojson


## Store Delivery Manifests

Generate one PDF per store (using `reportlab`) listing each of its trips with the delivery order and addresses, for drivers or for validating the routes.

In [18]:
import re

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

styles = getSampleStyleSheet()


def slugify(text):
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


trips_by_store = {}
for result in trip_results:
    trips_by_store.setdefault(result['store_idx'], []).append(result)

for store_idx, store_trip_results in trips_by_store.items():
    store_row = stores.iloc[store_idx]

    elements = [
        Paragraph(f"Delivery Manifest &mdash; {store_row['STORENAME']}", styles['Title']),
        Paragraph(store_row['ADDRESS'], styles['Normal']),
        Spacer(1, 16),
    ]

    for result in sorted(store_trip_results, key=lambda r: r['trip']):
        route = result['route']
        stops = [address_lookup[step['id']] for step in route['steps'] if step['type'] == 'job']

        elements.append(Paragraph(
            f"Trip {result['trip']} &mdash; {round(route['distance'] / 1000, 2)} km, "
            f"{round(route['duration'] / 60, 1)} min",
            styles['Heading2'],
        ))

        table_data = [['#', 'Address']] + [[str(i + 1), addr] for i, addr in enumerate(stops)]
        table = Table(table_data, colWidths=[30, 420])
        table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('FONTSIZE', (0, 0), (-1, -1), 9),
        ]))
        elements.append(table)
        elements.append(Spacer(1, 20))

    filename = f"manifest_{slugify(store_row['ADDRESS'])}.pdf"
    SimpleDocTemplate(str(output_dir / filename), pagesize=letter).build(elements)

print(f'Saved {len(trips_by_store)} manifest PDFs to {output_dir.resolve()}')

Saved 9 manifest PDFs to C:\Users\ujava\Desktop\claude_code_geospatial\outputs\route_optimization
